# p4a Session 划分

## 1. 划分依据

目前 sessions 之间差距主要在 systemprompt 拼接处就已经不同，因此无法做到拿全量的数据进行研究，而是需要按照 systemprompt 处分叉的依据来划分组，最后找到不同组来实现实验

systemPrompt 由多个块拼成，各块变化周期不同。跨 run 前缀共享到哪个字节，取决于最靠前的那个不同的块。所以类标签按块定义，不按整段哈希。

| 轴 | 含义 | 位置 | 来源列 |
|---|---|---|---|
| A | 工具配置 | systemPrompt 之前，~18,700 tok | `s3.tools_tok_rel_ref` |
| B | harness 版本 | systemPrompt 前半 + 尾部 | `s0b.axis_harness` |
| C | 时间戳 | systemPrompt 中部 | `s0b.axis_timestamp` |
| D | 项目目录树 | 时间戳之后 | `s0b.axis_tree` |
| E | 投递形态 | 首条 user message | `s0b.delivery` |
| F | 路径布局 | 首条 user message | `s0b.pointer_layout` |

## 2. 版本

统计结果并不来源于原本的 sessions ，而是一组数据处理脚本，以下为数据处理脚本版本控制

In [ ]:
import nbio
import pandas as pd
import plotly.express as px

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
nbio.banner()

In [ ]:
b   = nbio.load("s0b")
sb  = nbio.summary("s0b")
b["day"] = pd.to_datetime(b.created_at).dt.tz_convert("Asia/Shanghai").dt.date
b.shape

## 3. systemPrompt 块构成

按块统计互异内容数。互异 = 1 的块在全语料里逐字相同。

In [ ]:
blk = (b.block_md5.apply(pd.Series).nunique()
       .rename("互异内容").to_frame())
blk["出现份数"] = b.block_md5.apply(pd.Series).notna().sum()
blk["均长"]     = b.block_chars.apply(pd.Series).mean().round().astype("Int64")
blk["起始偏移p50"] = b.block_offset.apply(pd.Series).median().round().astype("Int64")
blk.sort_values("起始偏移p50").astype({"出现份数": "Int64"})

互异 = 1 的块是全语料的公共前缀，任何 run 都能复用。`Date and Time` 每 run 一变，`Working Directory` 有 10 个取值。`Language` / `Context Management` 只出现在一部分 session 里，所以 `axis_harness` 把块名也算进哈希。

## 4. 轴 B — harness 版本

用户在这一个月里升级过 kimi 的 CLI。`axis_harness` = 除时间戳 / 目录树 / AGENTS.md / skills 清单 / 机器属性外的全部块，按块名排序后取哈希。

In [ ]:
for a in ["axis_harness", "axis_agents", "axis_skills", "axis_machine",
          "axis_tree", "axis_timestamp"]:
    d = sb["axes"][a]
    print(f"{a:14} 互异 {d['n_distinct']:>5}  类大小 {d['sizes'][:12]}")

### harness、AGENTS.md、skills 划分一致

三者各 3 个取值，划分一致。用交叉表验证，不靠大小相同推断。版本号 V1/V2/V3 按首次出现时间排定，写在 `s0b` 产物里。

In [ ]:
ct = pd.crosstab(b.axis_harness, [b.axis_agents, b.axis_skills])
print(ct.to_string())
print()
assert (ct.gt(0).sum(axis=1) == 1).all(), "harness 与 agents/skills 并非完全共变"
print("✓ 每个 harness 取值只对应一组 (agents, skills)：三者完全共变")

# 版本号由 s0b 按首次出现时间排定，notebook 不重排
pd.DataFrame(sb["harness_versions"]).T

### 哪些块把版本分开

In [ ]:
bm  = b.block_md5.apply(pd.Series)
ver = b.harness_version
tab = pd.DataFrame({
    "版本间互异": bm.groupby(ver).first().nunique(),
    "版本内互异_max": bm.groupby(ver).nunique().max(),
    "出现于版本": bm.notna().groupby(ver).any().apply(
        lambda c: ", ".join(sorted(ver.unique()[[*range(len(c))]][c.values])), axis=0),
})
tab.reindex(bm.columns).sort_values(["版本间互异", "版本内互异_max"], ascending=False)

bm  = b.block_md5.apply(pd.Series)
ver = b.harness_version
tab = pd.DataFrame({
    "版本间互异": bm.groupby(ver).first().nunique(),
    "版本内互异_max": bm.groupby(ver).nunique().max(),
    "出现于版本": bm.notna().groupby(ver).any().apply(
        lambda c: ", ".join(sorted(ver.unique()[[*range(len(c))]][c.values])), axis=0),
})
tab.reindex(bm.columns).sort_values(["版本间互异", "版本内互异_max"], ascending=False)

### 对 s3 的影响

带 `llm.tools_snapshot` 的 60 份是 s3 复现器唯一的 ground truth。其余 4023 份的工具块由 Δ oracle 反推。

In [ ]:
s3 = nbio.load("s3")
gt = set(s3.loc[s3.matches_known_candidate.notna(), "sid"])
g  = b[b.sid.isin(gt)].harness_version.value_counts()
print(f"有 tools_snapshot 的 session: {len(gt)} 份，落在版本 {dict(g)}")
only = g.index[0]
tot  = (b.harness_version == only).sum()
print(f"版本 {only} 全体: {tot} 份")
print(f"覆盖率: {g.iloc[0]}/{tot} = {g.iloc[0] / tot:.0%}，其余版本 {(b.harness_version != only).sum()} 份未验证")

## 5. 轴 C — 时间戳

4074 个互异值。问题不在它变，在它的位置。

In [ ]:
cp = sb["timestamp_poisoning"]
print(f"时间戳块起始偏移  min {cp['timestamp_offset']['min']}  "
      f"p50 {cp['timestamp_offset']['p50']}  max {cp['timestamp_offset']['max']}")
print(f"其后残留字符    min {cp['poisoned_tail_chars']['min']}  "
      f"p50 {cp['poisoned_tail_chars']['p50']}  max {cp['poisoned_tail_chars']['max']}")
print(f"systemPrompt 均长 {b.sysprompt_chars.mean():.0f}")
print(f"→ 时间戳切在 {cp['timestamp_offset']['p50'] / b.sysprompt_chars.median():.0%} 处")

In [ ]:
tail = b.block_offset.apply(pd.Series)
order = tail.median().sort_values()
fig = px.bar(
    x=order.values, y=order.index, orientation="h",
    title="systemPrompt 各块的起始偏移（中位数）——红线是时间戳",
    labels={"x": "字符偏移", "y": ""}, height=460,
)
fig.add_vline(x=cp["timestamp_offset"]["p50"], line_color="crimson", line_width=2)
fig.update_yaxes(categoryorder="array", categoryarray=list(order.index)[::-1])
fig.show()

时间戳之后的块（目录树、AGENTS.md、skills 清单、Ultimate Reminders）在同一类里跨 run 逐字稳定，但因为排在时间戳后面而无法复用。

把时间戳移到 systemPrompt 末尾，类内共享前缀从 ~62% 升到接近 100%，语义不变。轴 B/D/E/F 的分歧要改动 workload 才能消除，轴 C 不用。

## 6. 轴 D — 项目目录树

用户在运行时修改过项目目录，而 kimi 的 systemPrompt 内嵌项目根的两层目录树。操作者改动目录即触发一次前缀断开。

In [ ]:
tree = (b.groupby("axis_tree")
         .agg(n=("sid", "size"), first=("day", "min"), last=("day", "max"),
              n_days=("day", "nunique"), ver=("harness_version", lambda s: sorted(set(s))))
         .sort_values("n", ascending=False))
tree

每个树状态占一段连续、几乎不重叠的日期区间。目录状态是一条单向推进的时间线。

### 树的变化

`s0b_summary.json` 的 `tree_bodies` 存了 10 份互异树的全文。下表是行的出现矩阵。

In [ ]:
bodies = sb["tree_bodies"]
order  = list(tree.index)                      # 按类大小排
rows_  = {t: set(bodies[t].split(chr(10))) for t in order}
allln  = [l for l in bodies[order[0]].split(chr(10))]
for t in order[1:]:
    for l in bodies[t].split(chr(10)):
        if l not in allln:
            allln.append(l)

mat = pd.DataFrame({t: [l in rows_[t] for l in allln] for t in order}, index=allln)
changing = mat[mat.sum(axis=1) < len(order)]   # 不是所有树都有的行
print(f"树共 {len(allln)} 种行，其中 {len(changing)} 种只出现在部分状态里")
changing.replace({True: "●", False: "·"})

列是树状态（按 run 数降序），行是树里的一行，`●` 表示存在。某行从 `·` 变 `●` 就是用户新建了那个目录，此前所有 run 的前缀在这一行断开。

In [ ]:
tl = pd.crosstab(b.day, b.axis_tree)
fig = px.imshow(tl.T, aspect="auto", color_continuous_scale="Blues",
                title="目录树状态 × 日期（每格 = 当天落在该树状态的 run 数）",
                labels=dict(x="日期", y="axis_tree", color="run 数"), height=380)
fig.show()

## 7. 轴 E — 投递形态

这一轴在首条 user message，与 systemPrompt 无关。同一个 workload 有两种把任务送进上下文的方式。

In [ ]:
pd.DataFrame(sb["prompt_chars_by_delivery"]).T

In [ ]:
fig = px.box(b[b.delivery != "other"], x="delivery", y="prompt_chars",
             log_y=True, points=False, height=380,
             title="首条 user message 的长度（对数轴）",
             labels={"delivery": "投递形态", "prompt_chars": "字符数"})
fig.show()

- **指针型**：prompt 里只写任务文件的路径（p50 378 字符），内容由 agent 自己 `Read`。首条 user message 跨 run 近乎逐字相同。
- **内联型**：整份 spec 塞进首条 user message，最长 95,512 字符。前缀在第一条用户消息就断开。

### family 编码了两件事

`wire.classify_family()` 判 family 用的就是这两个前缀，所以 family 和 delivery 现在是 1:1。但 family 说的是任务类型（抽取 / 修复），delivery 说的是投递方式。二者重合是这份语料的历史巧合，修复任务同样可以走指针型。

In [ ]:
pd.Series(sb["delivery_vs_family"]).rename("n").to_frame()

## 8. 轴 F — 路径布局

In [ ]:
print(pd.Series(sb["pointer_layout"]).rename("n").to_frame().to_string())
print()
print(pd.Series(sb["pointer_target"]).rename("n").to_frame().to_string())
print()
print("prompt 里出现的 PID 年份：", sb["pid_year"])

In [ ]:
pt = b[b.pointer_layout.notna()]
fig = px.histogram(pt, x="day", color="pointer_layout", height=360,
                   title="指针路径布局的迁移",
                   labels={"day": "日期", "count": "run 数"})
fig.show()

用户在 7 月中途把目录从 `layer4/<PID>/` 迁到 `layer4/2026/acl/<PID>/`，PID 年份同时从 `2025.acl-*` 换成 `2026.acl-*`。这一轴只影响首条 user message 末端的几十个字符，损耗最小。

## 9. 轴 A — 工具配置

工具块渲染在 systemPrompt 之前，约 18,700 tokens，是前缀里最大的一块。变体由 s3 的 Δ oracle 解出。它与 systemPrompt 的类互相正交，要分开记。

In [ ]:
j = b.merge(s3[["sid", "tools_tok_rel_ref"]], on="sid", how="left")
jj = j[j.tools_tok_rel_ref.notna()]
ct = pd.crosstab(jj.tools_tok_rel_ref.astype(int), jj.sysprompt_chars)
print(f"工具组 {ct.shape[0]} 个 × sysprompt 类 {ct.shape[1]} 个")
print(f"一个工具组最多横跨 {ct.gt(0).sum(axis=1).max()} 个 sysprompt 类")
print(f"一个 sysprompt 类最多横跨 {ct.gt(0).sum(axis=0).max()} 个工具组")
ct

## 10. 类标签

跨 run 前缀能否对齐由四轴联合决定（不含时间戳）。这个联合类有现成的代理：`s0` 早就在记的 `sysprompt_chars`。

In [ ]:
jc = sb["joint_classes"]
print(f"(harness, agents, skills, tree) 四轴联合：{jc['n']} 类")
print()
bij = sb["sysprompt_chars_is_class_label"]
print(f"与 sysprompt_chars 双射: {bij['bijective']}")
print(f"  {bij['n_lengths']} 个长度 ↔ {bij['n_classes']} 个类")
assert bij["bijective"], f"不再是双射：{bij['lengths_mapping_to_multiple_classes']}"
print()
print("→ sysprompt_chars 就是精确的类标签，下游不必重扫语料。")

前缀类记为三元组：

$$
\text{class} = \bigl(\underbrace{\text{tools\_}\Delta}_{18},\ \underbrace{\text{sysprompt\_chars}}_{10\,=\,\text{harness}\times\text{tree}},\ \underbrace{\text{delivery}}_{2}\bigr)
$$

时间戳不进类标签。它在每一类内部都打掉 systemPrompt 的后半段，是所有类共有的固定损耗。

In [ ]:
cls = (b.assign(tools=j.tools_tok_rel_ref)
        .groupby(["tools", "sysprompt_chars", "delivery"], dropna=False)
        .size().rename("n").reset_index().sort_values("n", ascending=False))
print(f"实际出现的三元组类：{len(cls)} 个，最大类 {cls.n.iloc[0]} 份，"
      f"单份类 {(cls.n == 1).sum()} 个")
cls.head(20)

## 11. 观察变量的选择

分组是为了挑出能做实验的组。忽略时间戳（轴 C，它是所有组共有的固定损耗）之后，先看剩下五轴里哪些是真的可以变的。

In [ ]:
s0 = nbio.load("s0")
inc = (b.merge(s3[["sid", "tools_tok_rel_ref"]], on="sid", how="left")
        .merge(s0[["sid", "included"]], on="sid")
        .query("included"))
inc["day"] = pd.to_datetime(inc.created_at).dt.tz_convert("Asia/Shanghai").dt.date
print(f"纳入集 {len(inc)} 份")
pd.crosstab(inc.harness_version, inc.delivery)

轴 B 和轴 E 在这份语料里是同一件事：V1 几乎全是内联，V2 和 V3 全是指针。用户在 06-22 一次性换掉了 harness、投递方式和任务类型，三者无法分离，只能整体固定。

所以**控制变量取 V2 + pointer**，剩下**轴 A（工具配置）和轴 D（目录树）作为观察变量**。

In [ ]:
core = inc[(inc.harness_version == "V2") & (inc.delivery == "pointer")]
cell = (core.groupby(["tools_tok_rel_ref", "axis_tree", "sysprompt_chars"], dropna=False)
            .agg(n=("sid", "size"), first=("day", "min"), last=("day", "max"))
            .sort_values("n", ascending=False))
print(f"V2 + pointer 共 {len(core)} 份，占纳入集 {len(core) / len(inc):.1%}，落在 {len(cell)} 个单元")
print(f"前 7 个单元覆盖 {cell.n.head(7).sum()} 份")
cell[cell.n >= 50]

### 交叉设计是现成的

上表里 `(-3882, 83a5abfa)` 那 530 份是两个方向的交点：

- **固定目录树 `83a5abfa`，变工具配置** → 隔离轴 A
- **固定工具配置 `-3882`，变目录树** → 隔离轴 D

两个方向都有几百份，不需要额外采样。

In [ ]:
A = cell.reset_index().query("axis_tree == '83a5abfa'").nlargest(3, "n")
D = cell.reset_index().query("tools_tok_rel_ref == -3882").nlargest(3, "n")
print("固定目录树 83a5abfa，变工具配置：")
print(A[["tools_tok_rel_ref", "n", "first", "last"]].to_string(index=False))
print()
print("固定工具配置 -3882，变目录树：")
print(D[["axis_tree", "sysprompt_chars", "n", "first", "last"]].to_string(index=False))

### 那 60 份的定位是校准集

V3 的 60 份是唯一带 `llm.tools_snapshot` 的，工具 schema 逐字已知，可以用来验证 Δ oracle 和 s3 复现器。但它们不适合当主要观察对象。

In [ ]:
v3 = inc[inc.harness_version == "V3"]
print(f"V3 共 {len(v3)} 份，日期 {v3.day.min()} .. {v3.day.max()}")
v3.groupby(["sysprompt_chars", "axis_tree", "tools_tok_rel_ref"], dropna=False).size().rename("n").to_frame()

三个原因：只有一天；57 份共用同一个工具配置，组内几乎没有可观察的变化；V3 是另一个 harness 版本，结论不能外推到 V2 那 3700 份。

**结论**

| | 取值 | 理由 |
|---|---|---|
| 控制 | harness = V2，delivery = pointer | 与 family 完全混淆，无法分离，只能整体固定 |
| 观察 | 轴 A 工具配置、轴 D 目录树 | V2 + pointer 内部唯一还在变的两轴 |
| 主组 | `(-3882, fb389653)` 961 份 | 最大的同质单元，单树单工具配置，连续 4 天 |
| 校准 | V3 那 60 份 | 工具 schema 逐字已知，用于验证复现器，不做分布结论 |

## 12. 复算

本 notebook 的数都来自 `s0b_summary.json` / `s3_render.jsonl`。下面复算它与 s0 的一致性，以及正文里写死的数。

In [ ]:
s0 = nbio.load("s0")
m = s0[["sid", "sysprompt_chars"]].merge(
    b[["sid", "sysprompt_chars"]], on="sid", suffixes=("_s0", "_s0b"))
assert (m.sysprompt_chars_s0 == m.sysprompt_chars_s0b).all(), "s0 与 s0b 的 systemPrompt 不一致"

checks = {
    "session 总数":        (len(b), 4083),
    "harness 版本":        (sb["axes"]["axis_harness"]["n_distinct"], 3),
    "目录树状态":          (sb["axes"]["axis_tree"]["n_distinct"], 10),
    "四轴联合类":          (sb["joint_classes"]["n"], 10),
    "sysprompt 长度取值":  (int(b.sysprompt_chars.nunique()), 10),
    "指针型":              (sb["delivery"]["pointer"], 3765),
    "内联型":              (sb["delivery"]["inline"], 306),
    "机器属性取值":        (sb["axes"]["axis_machine"]["n_distinct"], 1),
}
bad = {k: v for k, v in checks.items() if v[0] != v[1]}
for k, (got, want) in checks.items():
    print(f"{'ok ' if got == want else '✗  '} {k:20} 复算 {got:>6}  正文 {want:>6}")
assert not bad, f"与正文不一致：{bad}"